## **1. Projeto de Inteligência Artificial 2025/2026: Agentes para o Jogo PopOut**

**Faculdade de Ciências da Universidade do Porto (FCUP)** **Unidade Curricular:** Inteligência Artificial (CC2006)  
**Turma:** CC2006_PL1 | **Grupo:** 3  
**Docente:** Prof. Francesco Renna  

### **Equipe de Desenvolvimento**
| Número | Nome |
| :--- | :--- |
| **202300654** | Augusto Moreira |
| **202300276** | Guilherme Klippel |
| **202300916** | Yan Coelho |

---

### **Introdução e Contexto do Problema**

Este notebook documenta a conceção, implementação e análise de agentes de Inteligência Artificial para o jogo **PopOut**, uma variante estratégica e dinâmica do clássico *Connect-4* (Quatro em Linha). Tratando-se de um jogo de tabuleiro de soma nula e informação perfeita, o PopOut apresenta um desafio acrescido devido à sua mecânica de remoção de peças, que aumenta significativamente a complexidade e o fator de ramificação (*branching factor*) da árvore de jogo.

O objetivo deste projeto divide-se em duas abordagens fundamentais da IA:
1. **Procura Adversarial:** Implementação de um agente baseado em **Monte Carlo Tree Search (MCTS)** para atuar como um jogador de elite (*expert*).
2. **Aprendizagem Automática:** Utilização do MCTS para gerar conjuntos de dados de alta qualidade, servindo de base para o treino de um modelo de **Árvores de Decisão (algoritmo ID3)**, capaz de inferir regras táticas a partir das jogadas guardadas nos datasets.

### **Mecânicas e Regras Especiais**

A nossa implementação na classe base `PopOutGame` suporta o tabuleiro normal de 6x7 e introduz as seguintes ações:
* **Drop:** Colocação de uma peça no topo de uma coluna, caindo até à posição livre mais baixa.
* **PopOut:** Remoção de uma peça do próprio jogador na base (linha inferior) de uma coluna, fazendo com que todas as peças acima desçam uma posição.

Para garantir a robustez das simulações, foram rigorosamente implementadas as **três regras especiais** exigidas no guião:
* **Regra 1 (Vitória Simultânea):** Se uma jogada *PopOut* alinhar simultaneamente 4 peças para ambos os jogadores, a vitória é atribuída ao jogador que efetuou o movimento.
* **Regra 2 (Tabuleiro Cheio):** Caso a linha superior do tabuleiro esteja totalmente preenchida, o movimento *Drop* torna-se inválido, forçando o jogador a executar um *PopOut* (ou declarar derrota caso não tenha peças na base).
* **Regra 3 (Empate por Repetição):** O sistema monitoriza o histórico de estados (através de *hashes* do tabuleiro). Se o mesmo estado se repetir 3 vezes durante a partida, o jogo é declarado como empate.

---

#### Importes

In [4]:
%load_ext autoreload
%autoreload 2

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos locais
from src.game import PopOutGame, main_menu
from src.mcts import MCTS
from src.dataset_generator import run_batch_simulation

# Configurações globais
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
os.makedirs("datasets", exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## **2. Fase de Testes:**
Para garantir a modularidade e facilitar a futura implementação da árvore de procura, o projeto foi dividido em diferentes ficheiros:
* **`ui.py`**: Gere a interface visual e a renderização do tabuleiro no terminal.
* **`game.py`**: Contém a classe `PopOutGame`, responsável por validar os movimentos e aplicar as regras de vitória e empate.

Abaixo, inicializamos o menu principal para testar a mecânica base do tabuleiro.

In [ ]:
#MODO 2: HUMANO VS IA 
ia_desafio = MCTS(iterations=200, c=1.41, max_children=None)

#MODO 3: IA VS IA
# Estas são as duas IAs que vão jogar uma contra a outra na Opção 3.
ia_p1_visual = MCTS(iterations=1000, c=1.41, max_children=None)
ia_p2_visual = MCTS(iterations=15000, c=1.41, max_children=None)

main_menu(ia_std=ia_desafio, ia_p1=ia_p1_visual, ia_p2=ia_p2_visual)

## **3. Fase de Geração de Datasets**
Nesta etapa, o objetivo é utilizar o algoritmo **Monte Carlo Tree Search (MCTS)** para gerar exemplos de jogadas de alta qualidade que servirão de base para o treino da árvore de decisão **ID3**. O sistema executa simulações automáticas (IA vs IA), registando o estado do tabuleiro e a decisão estratégica tomada.

* **`run_batch_simulation`**: Função que coordena as partidas automáticas e garante a aplicação das regras de **PopOut**, **Vitória Dupla** e **Repetição**.
* **Identificação por Metadados**: O nome de cada ficheiro CSV é gerado dinamicamente com as configurações das IAs: **it** (iterações), **c** (exploração) e **mc** (max_children/largura).
* **Estrutura do Dataset**: Cada linha do ficheiro contém as 42 posições do tabuleiro (features), o turno atual e a jogada escolhida (target: coluna e tipo).

In [6]:
# Configuração das instâncias MCTS para treino
ia_elite = MCTS(iterations=100, c=1.41, max_children=None)
ia_caos = MCTS(iterations=1000, c=3.0, max_children=None)
ia_limitada = MCTS(iterations=1500, c=1.41, max_children=3)


# Execução dos lotes de simulação
run_batch_simulation(num_games=20, ia_1=ia_elite, ia_2=ia_elite)
#run_batch_simulation(num_games=20, ia_1=ia_elite, ia_2=ia_limitada)

print("Processo concluído. Ficheiros guardados na pasta 'datasets/'.")


[SISTEMA] A gerar lote de 20 jogos...
[ARQUIVO] datasets/P1_it100_c1.41_mc7_vs_P2_it100_c1.41_mc7.csv
> Jogo 1/20 guardado (ID: 60f4ebac | Vencedor: P1)
> Jogo 2/20 guardado (ID: 033fc5d1 | Vencedor: P2)
> Jogo 3/20 guardado (ID: 91775891 | Vencedor: P1)
> Jogo 4/20 guardado (ID: 27141b5e | Vencedor: P2)
> Jogo 5/20 guardado (ID: 840ee9d4 | Vencedor: P1)
> Jogo 6/20 guardado (ID: e95fd542 | Vencedor: P1)
> Jogo 7/20 guardado (ID: 736e523e | Vencedor: P2)
> Jogo 8/20 guardado (ID: 6c08ae56 | Vencedor: P2)
> Jogo 9/20 guardado (ID: 9228806f | Vencedor: P2)
> Jogo 10/20 guardado (ID: f992796d | Vencedor: P1)
> Jogo 11/20 guardado (ID: 115cf51c | Vencedor: P2)
> Jogo 12/20 guardado (ID: ca4a066b | Vencedor: P1)
> Jogo 13/20 guardado (ID: fe582f7c | Vencedor: P1)
> Jogo 14/20 guardado (ID: ca007a19 | Vencedor: P1)
> Jogo 15/20 guardado (ID: e8aa5139 | Vencedor: P2)
> Jogo 16/20 guardado (ID: c62bd575 | Vencedor: P1)
> Jogo 17/20 guardado (ID: e309c9da | Vencedor: P2)
> Jogo 18/20 guardado 

## **4. Fase de Análise Exploratória de Dados**

Esta fase é dedicada à auditoria visual do dataset gerado pelo **MCTS**. Antes de alimentar a Árvore de Decisão (**ID3**), precisamos de garantir que os dados são representativos. Com a introdução do rastreamento de vitórias (`winner` e `game_id`), a nossa análise deixa de focar apenas no *volume* de jogadas e passa a focar na *qualidade* estratégica.

### **Objetivos da Visualização**
* **Avaliação de Performance**: Verificar as taxas de vitória para validar o impacto dos parâmetros impostos à IA (ex: limitação do `max_children`).
* **Validação Estratégica**: Confirmar se a ocupação das colunas centrais dita o sucesso da partida.
* **Análise de Sucesso (Vencedor vs Perdedor)**: Contrastar diretamente o comportamento de quem ganha contra o comportamento de quem perde, isolando as táticas vitoriosas.

---

### **Dashboard de Performance e Estratégia**
O script de visualização gera um painel com 8 indicadores críticos, divididos entre comportamento geral e fatores de vitória:

1. **Resultados Finais das Partidas**: Exibe a taxa de vitórias (P1 vs P2) e empates, validando qual a configuração de IA mais dominante.
2. **Volume Total de Jogadas**: Revela a disparidade de movimentos totais executados por cada jogador ao longo do dataset.
3. **Distribuição de Colunas (P1 vs P2)**: Mostra a exploração lateral do tabuleiro, sem classificar o sucesso.
4. **Proporção Global (Drop vs Pop)**: Rácio de utilização da regra especial em relação à jogada clássica.
5. **Foco Estratégico (Vencedores vs Perdedores)**: O gráfico mais crítico. Mostra as colunas preferidas por quem acaba por ganhar o jogo em contraste com as escolhas de quem perde.
6. **Uso de Regras (Vencedores vs Perdedores)**: Analisa se a agressividade no uso do *PopOut* está diretamente correlacionada com a vitória.
7. **Heatmap de Ocupação**: Mapa térmico das 42 posições, mapeando as "Zonas Quentes" e trincheiras do tabuleiro.
8. **Zonas de PopOut (Vencedores vs Perdedores)**: Identifica em que colunas a regra de remoção de peça é taticamente mais letal e usada com sucesso.

In [ ]:
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

caminho_ficheiro = "datasets/P1_it2000_mc7_vs_P2_it1500_mc3.csv"
nome_do_ficheiro = os.path.basename(caminho_ficheiro)
df = pd.read_csv(caminho_ficheiro)

# Classificação das jogadas (Vencedor, Perdedor ou Empate)
df['is_winner'] = (df['p_turn'] == df['winner']) & (df['winner'] != 0)
df['status'] = df.apply(
    lambda r: 'Vencedor' if r['is_winner'] else ('Empate' if r['winner'] == 0 else 'Perdedor'), axis=1
)
df_jogos = df.drop_duplicates(subset=['game_id'])[['game_id', 'winner']]

fig, axes = plt.subplots(4, 2, figsize=(18, 24))
fig.suptitle(f"Análise Global e Estratégica: {nome_do_ficheiro}", fontsize=20, fontweight='bold', y=0.98)

# 1. Resultados Finais dos Jogos
sns.countplot(data=df_jogos, x='winner', hue='winner', ax=axes[0, 0], palette=['#95a5a6', '#3498db', '#e74c3c'], legend=False)
axes[0, 0].set_title("1. Resultados Finais das Partidas")
axes[0, 0].set_xticklabels(['Empate', 'Vitória P1 (X)', 'Vitória P2 (O)'])

# 2. Total de Movimentos (Geral)
sns.countplot(data=df, x='p_turn', hue='p_turn', ax=axes[0, 1], palette=['#3498db', '#e74c3c'], legend=False)
axes[0, 1].set_title("2. Volume Total de Jogadas (P1 vs P2)")

# 3. Escolha de Colunas (Geral)
sns.countplot(data=df, x='col', hue='p_turn', ax=axes[1, 0], palette=['#3498db', '#e74c3c'])
axes[1, 0].set_title("3. Distribuição de Colunas: P1 vs P2")

# 4. Rácio Drop vs Pop (Geral)
tipos = df['type'].value_counts()
axes[1, 1].pie(tipos, labels=tipos.index, autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1, 1].set_title("4. Proporção Global: Drop (d) vs Pop (p)")

# 5. Escolha de Colunas (Vencedor vs Perdedor)
df_decisivo = df[df['status'] != 'Empate']
sns.countplot(data=df_decisivo, x='col', hue='status', ax=axes[2, 0], palette=['#2ecc71', '#e67e22'])
axes[2, 0].set_title("5. Foco Estratégico: Vencedores vs Perdedores")

# 6. Uso da Mecânica (Vencedor vs Perdedor)
sns.countplot(data=df_decisivo, x='type', hue='status', ax=axes[2, 1], palette=['#2ecc71', '#e67e22'])
axes[2, 1].set_title("6. Uso de Regras: Vencedores vs Perdedores")

# 7. Heatmap de Ocupação (Geral)
colunas_tab = [f"pos_{i}" for i in range(42)]
ocupacao = (df[colunas_tab] > 0).mean().values.reshape(6, 7)
sns.heatmap(ocupacao, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[3, 0], cbar=False)
axes[3, 0].set_title("7. Mapa de Ocupação (Zonas Quentes)")

# 8. Onde ocorrem PopOuts? (Vencedor vs Perdedor)
pops = df_decisivo[df_decisivo['type'] == 'p']
if not pops.empty:
    sns.countplot(data=pops, x='col', hue='status', ax=axes[3, 1], palette=['#2ecc71', '#e67e22'])
    axes[3, 1].set_title("8. Zonas de PopOut: Vencedores vs Perdedores")
else:
    axes[3, 1].text(0.5, 0.5, "Sem jogadas PopOut", ha='center', va='center')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()